In [1]:
%load_ext autoreload
%autoreload 2

# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [2]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [3]:
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)

👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

In [4]:
features = [
    'wait_time',
    'expected_wait_time',
    'delay_vs_expected',
    'number_of_items',
    'number_of_sellers',
    'price',
    'freight_value'
]

orders[features].head()

,wait_time,expected_wait_time,delay_vs_expected,number_of_items,number_of_sellers,price,freight_value
0,8.436574,15.544063,0.0,1,1,29.99,8.72
1,13.782037,19.137766,0.0,1,1,118.70,22.76
2,9.394213,26.639711,0.0,1,1,159.90,19.22
3,13.208750,26.188819,0.0,1,1,45.00,27.20
4,2.873877,12.112049,0.0,1,1,19.90,8.72


🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [5]:
orders_standardized = orders.copy()
orders_standardized[features] = (orders[features] - orders[features].mean()) / orders[features].std()

orders_standardized[features].describe()

,wait_time,expected_wait_time,delay_vs_expected,number_of_items,number_of_sellers,price,freight_value
count,9.635300e+04,9.635300e+04,9.635300e+04,9.635300e+04,9.635300e+04,9.635300e+04,9.635300e+04
mean,-1.258068e-16,1.159251e-16,2.053762e-17,5.648768e-17,4.785229e-16,6.511569e-17,-1.317063e-16
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-1.267762e+00,-2.481313e+00,-1.621188e-01,-2.646571e-01,-1.125875e-01,-6.546565e-01,-1.058699e+00
25%,-6.089910e-01,-6.172652e-01,-1.621188e-01,-2.646571e-01,-1.125875e-01,-4.374896e-01,-4.148587e-01
50%,-2.443566e-01,-5.808503e-02,-1.621188e-01,-2.646571e-01,-1.125875e-01,-2.441845e-01,-2.604115e-01
75%,3.343922e-01,5.331759e-01,-1.621188e-01,-2.646571e-01,-1.125875e-01,6.385035e-02,5.732178e-02
max,2.070689e+01,1.500095e+01,4.042112e+01,3.709324e+01,3.224580e+01,6.412979e+01,8.244328e+01


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

In [6]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = orders_standardized[features]

vif_df = pd.DataFrame()
vif_df["feature"] = features
vif_df["vif"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

vif_df.sort_values('vif', ascending=False)

,feature,vif
0,wait_time,2.832725
2,delay_vs_expected,2.379695
6,freight_value,1.577027
1,expected_wait_time,1.443633
3,number_of_items,1.348236
5,price,1.204880
4,number_of_sellers,1.095442


## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [7]:
logit_one = smf.logit(
    formula='dim_is_one_star ~ ' + ' + '.join(features),
    data=orders_standardized
).fit()

logit_one.summary()

Optimization terminated successfully.
         Current function value: 0.273079
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:        dim_is_one_star   No. Observations:                96353
Model:                          Logit   Df Residuals:                    96345
Method:                           MLE   Df Model:                            7
Date:                Thu, 10 Sep 2026   Pseudo R-squ.:                  0.1461
Time:                        16:38:22   Log-Likelihood:                -26312.
converged:                       True   LL-Null:                       -30814.
Covariance Type:            nonrobust   LLR p-value:                     0.000
======================================================================================
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -2.4824      0.013   -189.580      0.000      -2.508      -2.457
wait_time              0.7956      0.019     41.530      0.000       0.758       0.833
expected_wait_time    -0.2484      0.016    -15.450      0.000      -0.280      -0.217
delay_vs_expected      0.1573      0.020      7.794      0.000       0.118       0.197
number_of_items        0.2645      0.011     24.977      0.000       0.244       0.285
number_of_sellers      0.1867      0.008     23.639      0.000       0.171       0.202
price                  0.0579      0.011      5.127      0.000       0.036       0.080
freight_value         -0.0422      0.013     -3.193      0.001      -0.068      -0.016
======================================================================================
"""

`Logit 5️⃣`

In [8]:
logit_five = smf.logit(
    formula='dim_is_five_star ~ ' + ' + '.join(features),
    data=orders_standardized
).fit()

logit_five.summary()

Optimization terminated successfully.
         Current function value: 0.636930
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:       dim_is_five_star   No. Observations:                96353
Model:                          Logit   Df Residuals:                    96345
Method:                           MLE   Df Model:                            7
Date:                Thu, 10 Sep 2026   Pseudo R-squ.:                 0.05787
Time:                        16:38:32   Log-Likelihood:                -61370.
converged:                       True   LL-Null:                       -65140.
Covariance Type:            nonrobust   LLR p-value:                     0.000
======================================================================================
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.3427      0.007     48.102      0.000       0.329       0.357
wait_time             -0.5389      0.012    -44.095      0.000      -0.563      -0.515
expected_wait_time     0.0943      0.008     11.206      0.000       0.078       0.111
delay_vs_expected     -0.3818      0.024    -16.046      0.000      -0.428      -0.335
number_of_items       -0.1447      0.008    -17.454      0.000      -0.161      -0.128
number_of_sellers     -0.1450      0.008    -18.550      0.000      -0.160      -0.130
price                  0.0167      0.008      2.201      0.028       0.002       0.032
freight_value          0.0180      0.009      2.071      0.038       0.001       0.035
======================================================================================
"""

💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [11]:
# Aşağıdaki cümlelerden doğru olanları aşağıdaki listeye kaydedin.

a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = [a]

In [9]:
comparison = pd.DataFrame({
    'logit_one': logit_one.params,
    'logit_five': logit_five.params,
    'p_one': logit_one.pvalues,
    'p_five': logit_five.pvalues
}).drop('Intercept')

comparison

,logit_one,logit_five,p_one,p_five
wait_time,0.795642,-0.538864,0.000000e+00,0.000000e+00
expected_wait_time,-0.248387,0.094258,7.578961e-54,3.805509e-29
delay_vs_expected,0.157306,-0.381834,6.489213e-15,6.114596e-58
number_of_items,0.264482,-0.144738,1.098261e-137,3.182792e-68
number_of_sellers,0.186682,-0.145023,1.520938e-123,8.191942e-77
price,0.057892,0.016725,2.941420e-07,2.775293e-02
freight_value,-0.042185,0.017958,1.409717e-03,3.839327e-02


🧪 __Kodunu Test Et__

In [12]:
from nbresult import ChallengeResult

result = ChallengeResult('logit',
    answers = your_answer
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/kaanu/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/kaanu/code/sprint-16/04-Seaborn-Regression/data-logit/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!